In [1]:
# Cell 1: Install required libraries and import everything
!pip install --upgrade numpy --quiet
!pip install transformers datasets optuna pyswarms deap torch --quiet

import numpy as np
print(f"NumPy version: {np.__version__}")

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW  # Using torch.optim.AdamW
from datasets import load_dataset
from sklearn.metrics import f1_score, accuracy_score
import time
import gc
import random
from copy import deepcopy
import json
import warnings
warnings.filterwarnings('ignore')

# For GPU memory tracking
import subprocess

# Set random seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"Memory Cached: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

# Fixed batch size
BATCH_SIZE = 32
MAX_LENGTH = 128  # Reduced for faster training during optimization

NumPy version: 2.4.6
Using device: cuda
GPU: Tesla T4
Memory Allocated: 0.00 GB
Memory Cached: 0.00 GB


In [2]:
# Cell 2: Load IMDB dataset with balanced splits, no overlap, no duplicates
print("Loading IMDB dataset...")

# Load full dataset with correct path
dataset = load_dataset("stanfordnlp/imdb")

# Function to remove duplicate texts while keeping first occurrence
def remove_duplicates(data):
    """Remove duplicate texts from dataset"""
    seen_texts = set()
    unique_indices = []

    for i, example in enumerate(data):
        text = example["text"]
        if text not in seen_texts:
            seen_texts.add(text)
            unique_indices.append(i)

    print(f"  Removed {len(data) - len(unique_indices)} duplicates from {len(data)} samples")
    return data.select(unique_indices)

# Function to balance dataset by label
def balance_dataset(data, n_samples):
    """Take n_samples total with equal positive and negative examples"""
    n_per_class = n_samples // 2

    pos_indices = [i for i, x in enumerate(data) if x["label"] == 1]
    neg_indices = [i for i, x in enumerate(data) if x["label"] == 0]

    # Shuffle indices
    random.shuffle(pos_indices)
    random.shuffle(neg_indices)

    # Take equal numbers from each class
    selected_pos = pos_indices[:n_per_class]
    selected_neg = neg_indices[:n_per_class]

    # Combine and shuffle
    all_indices = selected_pos + selected_neg
    random.shuffle(all_indices)

    return data.select(all_indices)

# Step 1: Remove duplicates from full train set
print("\nRemoving duplicates from training data...")
train_clean = remove_duplicates(dataset["train"])

# Step 2: Create balanced train set: 20,000 samples
print("\nCreating balanced training set (20,000 samples)...")
train_full = balance_dataset(train_clean, 20000)

# Step 3: Split into 90% train and 10% validation (balanced, no overlap)
print("\nSplitting into 90% train and 10% validation (balanced)...")
train_full = train_full.shuffle(seed=42)

# Split while maintaining balance
train_pos_indices = [i for i, x in enumerate(train_full) if x["label"] == 1]
train_neg_indices = [i for i, x in enumerate(train_full) if x["label"] == 0]

# 90% of each class for train
n_train_pos = int(len(train_pos_indices) * 0.9)
n_train_neg = int(len(train_neg_indices) * 0.9)

# These are index positions within train_full, NOT overlapping
train_indices = train_pos_indices[:n_train_pos] + train_neg_indices[:n_train_neg]
val_indices = train_pos_indices[n_train_pos:] + train_neg_indices[n_train_neg:]

# Shuffle indices
random.shuffle(train_indices)
random.shuffle(val_indices)

# Create final splits
train_dataset = train_full.select(train_indices)
val_dataset = train_full.select(val_indices)

# Step 4: Remove duplicates from test set and create balanced set
print("\nRemoving duplicates from test data...")
test_clean = remove_duplicates(dataset["test"])

# Also remove any texts that appear in training data
train_val_texts = set(train_dataset["text"]) | set(val_dataset["text"])
test_unique_indices = []
for i, example in enumerate(test_clean):
    if example["text"] not in train_val_texts:
        test_unique_indices.append(i)
print(f"  Removed {len(test_clean) - len(test_unique_indices)} test samples overlapping with train/val")
test_clean = test_clean.select(test_unique_indices)

print("\nCreating balanced test set (20,000 samples)...")
test_dataset = balance_dataset(test_clean, 20000)

# VERIFICATION
print(f"\n{'='*60}")
print(f"FINAL VERIFICATION:")
print(f"{'='*60}")

# Check sizes
print(f"\nTrain:      {len(train_dataset)} samples")
print(f"  Positive: {sum(1 for x in train_dataset if x['label'] == 1)}")
print(f"  Negative: {sum(1 for x in train_dataset if x['label'] == 0)}")
print(f"Validation: {len(val_dataset)} samples")
print(f"  Positive: {sum(1 for x in val_dataset if x['label'] == 1)}")
print(f"  Negative: {sum(1 for x in val_dataset if x['label'] == 0)}")
print(f"Test:       {len(test_dataset)} samples")
print(f"  Positive: {sum(1 for x in test_dataset if x['label'] == 1)}")
print(f"  Negative: {sum(1 for x in test_dataset if x['label'] == 0)}")

# Check overlap
train_texts = set(train_dataset["text"])
val_texts = set(val_dataset["text"])
test_texts = set(test_dataset["text"])

overlap_train_val = train_texts.intersection(val_texts)
overlap_train_test = train_texts.intersection(test_texts)
overlap_val_test = val_texts.intersection(test_texts)

print(f"\nOverlap Check:")
print(f"Train & Validation: {len(overlap_train_val)} samples",
      "✓ NO LEAKAGE!" if len(overlap_train_val) == 0 else "✗ WARNING!")
print(f"Train & Test:       {len(overlap_train_test)} samples",
      "✓ NO LEAKAGE!" if len(overlap_train_test) == 0 else "✗ WARNING!")
print(f"Val & Test:         {len(overlap_val_test)} samples",
      "✓ NO LEAKAGE!" if len(overlap_val_test) == 0 else "✗ WARNING!")

# Check balance
train_pos = sum(1 for x in train_dataset if x["label"] == 1)
train_neg = sum(1 for x in train_dataset if x["label"] == 0)
val_pos = sum(1 for x in val_dataset if x["label"] == 1)
val_neg = sum(1 for x in val_dataset if x["label"] == 0)
test_pos = sum(1 for x in test_dataset if x["label"] == 1)
test_neg = sum(1 for x in test_dataset if x["label"] == 0)

print(f"\nBalance Check:")
print(f"Train:      {train_pos}/{train_neg} (ratio: {train_pos/train_neg:.3f})")
print(f"Validation: {val_pos}/{val_neg} (ratio: {val_pos/val_neg:.3f})")
print(f"Test:       {test_pos}/{test_neg} (ratio: {test_pos/test_neg:.3f})")
print(f"{'='*60}")

Loading IMDB dataset...



Removing duplicates from training data...
  Removed 96 duplicates from 25000 samples

Creating balanced training set (20,000 samples)...

Splitting into 90% train and 10% validation (balanced)...

Removing duplicates from test data...
  Removed 199 duplicates from 25000 samples
  Removed 91 test samples overlapping with train/val

Creating balanced test set (20,000 samples)...

FINAL VERIFICATION:

Train:      18000 samples
  Positive: 9000
  Negative: 9000
Validation: 2000 samples
  Positive: 1000
  Negative: 1000
Test:       20000 samples
  Positive: 10000
  Negative: 10000

Overlap Check:
Train & Validation: 0 samples ✓ NO LEAKAGE!
Train & Test:       0 samples ✓ NO LEAKAGE!
Val & Test:         0 samples ✓ NO LEAKAGE!

Balance Check:
Train:      9000/9000 (ratio: 1.000)
Validation: 1000/1000 (ratio: 1.000)
Test:       10000/10000 (ratio: 1.000)


In [3]:
# Cell 3: Tokenize all three datasets
print("Loading tokenizer...")
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors=None  # Don't return tensors yet
    )

# Tokenize datasets
print("Tokenizing train dataset...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
print("Tokenizing validation dataset...")
val_dataset = val_dataset.map(tokenize_function, batched=True)
print("Tokenizing test dataset...")
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
columns = ["input_ids", "attention_mask", "label"]
train_dataset.set_format(type="torch", columns=columns)
val_dataset.set_format(type="torch", columns=columns)
test_dataset.set_format(type="torch", columns=columns)

print("\nTokenization complete!")

Loading tokenizer...
Tokenizing train dataset...
Tokenizing validation dataset...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing test dataset...

Tokenization complete!


In [4]:
# Run this at the beginning of your Colab notebook
!pip install av --quiet
!pip install torchvision --upgrade --quiet

# Then restart the runtime (Runtime -> Restart runtime)
# After restart, your code should work

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
libraft-cu12 26.2.0 requires cuda-toolkit[cublas,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cuda-python 12.9.6 requires cuda-bindings~=12.9.6, but you have cuda-bindings 13.3.1 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
libcuvs-cu12 26.2.0 requires cuda-toolkit[cublas,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
libcuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-

In [9]:
# Cell 5: Define training, evaluation, and metrics functions (with progress)
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

def get_gpu_memory_usage():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return 0

def reset_gpu_memory_stats():
    """Reset GPU memory stats"""
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

def evaluate_model(model, data_loader, desc="Evaluating"):
    """Evaluate model and return metrics"""
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0
    inference_times = []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc=desc, leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            # Measure inference time per batch
            start_time = time.time()
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            inference_times.append(time.time() - start_time)

            total_loss += outputs.loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    avg_loss = total_loss / len(data_loader)
    avg_inference_time = np.mean(inference_times)

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "f1": f1,
        "inference_time_per_batch": avg_inference_time
    }

def train_model(hyperparams, train_loader, val_loader, epochs, lr, weight_decay,
                warmup_ratio, dropout_rate, verbose=True):
    """Train DistilBERT with given hyperparameters and return metrics"""

    # Reset GPU memory tracking
    reset_gpu_memory_stats()
    torch.cuda.empty_cache()
    gc.collect()

    if verbose:
        print(f"Loading model with dropout={dropout_rate}...")

    # Initialize model with dropout rate
    model = DistilBertForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=2,
        dropout=dropout_rate,
        attention_dropout=dropout_rate
    ).to(device)

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if verbose:
        print(f"Total parameters: {total_params:,}")
        print(f"Trainable parameters: {trainable_params:,}")

    # Optimizer
    optimizer = AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    # Scheduler
    total_steps = len(train_loader) * epochs
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    if verbose:
        print(f"\nTraining for {epochs} epochs ({total_steps} total steps, {warmup_steps} warmup steps)...")
        print(f"Learning rate: {lr}, Weight decay: {weight_decay}")
        print("-" * 50)

    # Training
    training_start = time.time()

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False) if verbose else train_loader

        for batch_idx, batch in enumerate(progress_bar):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()

            # Update progress bar
            if verbose and isinstance(progress_bar, tqdm):
                progress_bar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.2e}'})

        avg_epoch_loss = epoch_loss / len(train_loader)
        if verbose:
            print(f"Epoch {epoch+1}/{epochs} - Avg Loss: {avg_epoch_loss:.4f}")

    training_time = time.time() - training_start

    if verbose:
        print(f"\nTraining completed in {training_time:.2f} seconds")
        print("-" * 50)
        print("Evaluating on validation set...")

    # Evaluate on validation set
    val_metrics = evaluate_model(model, val_loader, desc="Validating")

    # Get GPU memory usage
    gpu_memory = get_gpu_memory_usage()

    if verbose:
        print(f"Validation Loss: {val_metrics['loss']:.4f}")
        print(f"Validation Accuracy: {val_metrics['accuracy']:.4f}")
        print(f"Validation F1: {val_metrics['f1']:.4f}")
        print(f"GPU Memory: {gpu_memory:.2f} MB")

    # Clean up
    del model
    torch.cuda.empty_cache()
    gc.collect()

    return {
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_f1": val_metrics["f1"],
        "training_time": training_time,
        "inference_time_per_batch": val_metrics["inference_time_per_batch"],
        "gpu_memory_mb": gpu_memory
    }

print("Functions defined with progress tracking!")

Train batches: 563, Val batches: 63, Test batches: 625
Functions defined with progress tracking!


In [10]:
# Cell 6: Train baseline model with default hyperparameters
print("Training baseline model with default hyperparameters...")
print(f"{'='*60}")

# Default hyperparameters
baseline_hyperparams = {
    "learning_rate": 2e-5,
    "epochs": 5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "dropout_rate": 0.1
}

print("Hyperparameters:")
for k, v in baseline_hyperparams.items():
    print(f"  {k}: {v}")

# Train baseline model
baseline_results = train_model(
    hyperparams=baseline_hyperparams,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=baseline_hyperparams["epochs"],
    lr=baseline_hyperparams["learning_rate"],
    weight_decay=baseline_hyperparams["weight_decay"],
    warmup_ratio=baseline_hyperparams["warmup_ratio"],
    dropout_rate=baseline_hyperparams["dropout_rate"],
    verbose=True
)

print(f"\n{'='*60}")
print("BASELINE RESULTS (Validation Set):")
print(f"{'='*60}")
print(f"Validation Loss:      {baseline_results['val_loss']:.4f}")
print(f"Validation Accuracy:  {baseline_results['val_accuracy']:.4f}")
print(f"Validation F1-Score:  {baseline_results['val_f1']:.4f}")
print(f"Training Time:        {baseline_results['training_time']:.2f} seconds")
print(f"Inference Time/Batch: {baseline_results['inference_time_per_batch']:.4f} seconds")
print(f"GPU Memory Used:      {baseline_results['gpu_memory_mb']:.2f} MB")
print(f"{'='*60}")

# Estimate optimization time
estimated_time_20_runs = baseline_results['training_time'] * 20
print(f"\n⚠ ESTIMATE: 20 runs would take ~{estimated_time_20_runs/60:.1f} minutes")
print(f"   With 3 algorithms (PS + GA + Grid Search): ~{estimated_time_20_runs*3/60:.1f} minutes")
print(f"\nConsider reducing epochs or dataset size if this is too long.")

Training baseline model with default hyperparameters...
Hyperparameters:
  learning_rate: 2e-05
  epochs: 5
  weight_decay: 0.01
  warmup_ratio: 0.1
  dropout_rate: 0.1
Loading model with dropout=0.1...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters: 66,955,010
Trainable parameters: 66,955,010

Training for 5 epochs (2815 total steps, 281 warmup steps)...
Learning rate: 2e-05, Weight decay: 0.01
--------------------------------------------------


Epoch 1/5 - Avg Loss: 0.4361


Epoch 2/5 - Avg Loss: 0.2637


Epoch 3/5 - Avg Loss: 0.1698


Epoch 4/5 - Avg Loss: 0.1016


Epoch 5/5 - Avg Loss: 0.0651

Training completed in 1102.18 seconds
--------------------------------------------------
Evaluating on validation set...


Validation Loss: 0.4840
Validation Accuracy: 0.8730
Validation F1: 0.8730
GPU Memory: 2094.94 MB

BASELINE RESULTS (Validation Set):
Validation Loss:      0.4840
Validation Accuracy:  0.8730
Validation F1-Score:  0.8730
Training Time:        1102.18 seconds
Inference Time/Batch: 0.0065 seconds
GPU Memory Used:      2094.94 MB

⚠ ESTIMATE: 20 runs would take ~367.4 minutes
   With 3 algorithms (PS + GA + Grid Search): ~1102.2 minutes

Consider reducing epochs or dataset size if this is too long.


In [11]:
# Cell 7: Define unified hyperparameter search space for all algorithms
print("=" * 60)
print("HYPERPARAMETER SEARCH SPACE")
print("=" * 60)

# Search space definition
# Format: [lower_bound, upper_bound, scale_type]
search_space = {
    "learning_rate": {
        "low": 1e-6,
        "high": 5e-5,
        "scale": "log",
        "description": "Learning rate for AdamW optimizer"
    },
    "epochs": {
        "low": 2,
        "high": 4,
        "scale": "int",
        "description": "Number of training epochs"
    },
    "weight_decay": {
        "low": 1e-5,
        "high": 1e-1,
        "scale": "log",
        "description": "L2 regularization strength"
    },
    "warmup_ratio": {
        "low": 0.0,
        "high": 0.2,
        "scale": "linear",
        "description": "Fraction of steps for LR warmup"
    },
    "dropout_rate": {
        "low": 0.05,
        "high": 0.4,
        "scale": "linear",
        "description": "Dropout probability for attention and hidden layers"
    }
}

# Print search space
for param, bounds in search_space.items():
    print(f"\n{param}:")
    print(f"  Range: [{bounds['low']}, {bounds['high']}]")
    print(f"  Scale: {bounds['scale']}")
    print(f"  {bounds['description']}")

# Extract bounds for optimization algorithms
param_names = list(search_space.keys())
lb = [search_space[p]["low"] for p in param_names]   # Lower bounds
ub = [search_space[p]["high"] for p in param_names]   # Upper bounds
scales = [search_space[p]["scale"] for p in param_names]

print(f"\n{'='*60}")
print("Bounds arrays for algorithms:")
print(f"  Parameters: {param_names}")
print(f"  Lower bounds: {lb}")
print(f"  Upper bounds: {ub}")
print(f"  Scales: {scales}")
print(f"{'='*60}")

# Number of evaluations per algorithm
N_EVALUATIONS = 20
print(f"\n⚠ Each algorithm will run {N_EVALUATIONS} evaluations")
print(f"   Total across 3 algorithms: {N_EVALUATIONS * 3} training runs")

HYPERPARAMETER SEARCH SPACE

learning_rate:
  Range: [1e-06, 5e-05]
  Scale: log
  Learning rate for AdamW optimizer

epochs:
  Range: [2, 4]
  Scale: int
  Number of training epochs

weight_decay:
  Range: [1e-05, 0.1]
  Scale: log
  L2 regularization strength

warmup_ratio:
  Range: [0.0, 0.2]
  Scale: linear
  Fraction of steps for LR warmup

dropout_rate:
  Range: [0.05, 0.4]
  Scale: linear
  Dropout probability for attention and hidden layers

Bounds arrays for algorithms:
  Parameters: ['learning_rate', 'epochs', 'weight_decay', 'warmup_ratio', 'dropout_rate']
  Lower bounds: [1e-06, 2, 1e-05, 0.0, 0.05]
  Upper bounds: [5e-05, 4, 0.1, 0.2, 0.4]
  Scales: ['log', 'int', 'log', 'linear', 'linear']

⚠ Each algorithm will run 20 evaluations
   Total across 3 algorithms: 60 training runs


In [12]:
# Cell 8: Lightweight training function for hyperparameter optimization
def train_model_fast(epochs, lr, weight_decay, warmup_ratio, dropout_rate):
    """Train DistilBERT silently and return metrics dictionary"""

    # Reset GPU memory tracking
    reset_gpu_memory_stats()
    torch.cuda.empty_cache()
    gc.collect()

    # Initialize model
    model = DistilBertForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=2,
        dropout=dropout_rate,
        attention_dropout=dropout_rate
    ).to(device)

    # Optimizer
    optimizer = AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    # Scheduler
    total_steps = len(train_loader) * epochs
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    # Training
    training_start = time.time()

    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            scheduler.step()

    training_time = time.time() - training_start

    # Evaluate on validation set
    val_metrics = evaluate_model(model, val_loader, desc="")

    # GPU memory
    gpu_memory = get_gpu_memory_usage()

    # Clean up
    del model
    torch.cuda.empty_cache()
    gc.collect()

    return {
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_f1": val_metrics["f1"],
        "training_time": training_time,
        "inference_time_per_batch": val_metrics["inference_time_per_batch"],
        "gpu_memory_mb": gpu_memory
    }

# Quick test
print("Testing fast training function...")
test_result = train_model_fast(epochs=1, lr=2e-5, weight_decay=0.01, warmup_ratio=0.1, dropout_rate=0.1)
print(f"Test run - Loss: {test_result['val_loss']:.4f}, Acc: {test_result['val_accuracy']:.4f}, Time: {test_result['training_time']:.2f}s")
print("✓ Fast training function ready!")

Testing fast training function...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Test run - Loss: 0.2932, Acc: 0.8760, Time: 222.43s
✓ Fast training function ready!


In [ ]:
# Cell 7+8: Search Space Definition + All Functions + Fast Training
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm

# ============================================
# GPU UTILITY FUNCTIONS
# ============================================
def get_gpu_memory_usage():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return 0

def reset_gpu_memory_stats():
    """Reset GPU memory stats"""
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

# ============================================
# EVALUATION FUNCTION
# ============================================
def evaluate_model(model, data_loader, desc="Evaluating"):
    """Evaluate model and return metrics"""
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0
    inference_times = []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc=desc, leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            start_time = time.time()
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            inference_times.append(time.time() - start_time)

            total_loss += outputs.loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    avg_loss = total_loss / len(data_loader)
    avg_inference_time = np.mean(inference_times)

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "f1": f1,
        "inference_time_per_batch": avg_inference_time
    }

# ============================================
# HYPERPARAMETER SEARCH SPACE
# ============================================
print("=" * 60)
print("HYPERPARAMETER SEARCH SPACE")
print("=" * 60)

search_space = {
    "learning_rate":  {"low": 1e-6, "high": 5e-5,  "scale": "log"},
    "epochs":         {"low": 2,    "high": 4,     "scale": "int"},
    "weight_decay":   {"low": 1e-5, "high": 1e-1,  "scale": "log"},
    "warmup_ratio":   {"low": 0.0,  "high": 0.2,   "scale": "linear"},
    "dropout_rate":   {"low": 0.05, "high": 0.4,   "scale": "linear"}
}

param_names = list(search_space.keys())
lb = [search_space[p]["low"] for p in param_names]
ub = [search_space[p]["high"] for p in param_names]
scales = [search_space[p]["scale"] for p in param_names]

for param, bounds in search_space.items():
    print(f"  {param}: [{bounds['low']}, {bounds['high']}] ({bounds['scale']})")

N_EVALUATIONS = 20
print(f"\nEvaluations per algorithm: {N_EVALUATIONS}")
print(f"Total across 3 algorithms: {N_EVALUATIONS * 3}")
print("=" * 60)

# ============================================
# FAST TRAINING FUNCTION
# ============================================
def train_model_fast(epochs, lr, weight_decay, warmup_ratio, dropout_rate):
    """Train DistilBERT silently and return metrics dictionary"""

    reset_gpu_memory_stats()
    torch.cuda.empty_cache()
    gc.collect()

    model = DistilBertForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=2,
        dropout=dropout_rate,
        attention_dropout=dropout_rate
    ).to(device)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_steps = len(train_loader) * epochs
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    training_start = time.time()

    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

    training_time = time.time() - training_start
    val_metrics = evaluate_model(model, val_loader, desc="")
    gpu_memory = get_gpu_memory_usage()

    del model
    torch.cuda.empty_cache()
    gc.collect()

    return {
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_f1": val_metrics["f1"],
        "training_time": training_time,
        "inference_time_per_batch": val_metrics["inference_time_per_batch"],
        "gpu_memory_mb": gpu_memory
    }

# Quick test
print("\nTesting fast training function...")
test_result = train_model_fast(epochs=1, lr=2e-5, weight_decay=0.01, warmup_ratio=0.1, dropout_rate=0.1)
print(f"Test run - Loss: {test_result['val_loss']:.4f}, Acc: {test_result['val_accuracy']:.4f}, Time: {test_result['training_time']:.2f}s")
print("✓ All functions ready!")